# Basic RAG Pipeline & Retrieval Performance Analysis

## Objective

Build a basic Retrieval-Augmented Generation (RAG) system using company documentation.

The project demonstrates:

- Document loading
- Document chunking
- Embeddings
- Vector similarity search
- Semantic retrieval
- Retrieval-Augmented Generation
- Retrieval performance analysis
- Chunk-size comparison

## RAG Architecture

Company Documentation  
↓  
Document Chunking  
↓  
Embeddings  
↓  
Vector Search  
↓  
Semantic Retrieval  
↓  
Context + Question  
↓  
Answer Generation  

## Vector Databases

Vector databases store numerical representations called embeddings and allow semantically similar information to be retrieved efficiently.

Popular technologies include:

- ChromaDB
- FAISS

This experiment implements vector similarity retrieval directly using Sentence Transformers and cosine similarity to demonstrate the core retrieval process.

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

print("===== BASIC RAG PIPELINE =====")

with open("company_documentation.txt", "r", encoding="utf-8") as file:
    document = file.read().strip()

if not document:
    raise ValueError("company_documentation.txt is empty.")

print("\nDocument loaded successfully.")
print("Document length:", len(document), "characters")
print("Document words:", len(document.split()))

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

questions = [
    "What is HearMe?",
    "How does HearMe process a customer call?",
    "How is ASR performance evaluated?",
    "How can RAG improve HearMe?"
]

chunk_sizes = [50, 100, 150]

def create_chunks(text, chunk_size):
    words = text.split()

    if len(words) == 0:
        return []

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size]).strip()

        if chunk:
            chunks.append(chunk)

    return chunks

def retrieve_context(question, chunks):

    if len(chunks) == 0:
        raise ValueError("No document chunks were created.")

    chunk_embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True
    )

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    if chunk_embeddings.ndim == 1:
        chunk_embeddings = chunk_embeddings.reshape(1, -1)

    if question_embedding.ndim == 1:
        question_embedding = question_embedding.reshape(1, -1)

    similarities = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    best_index = np.argmax(similarities)

    return chunks[best_index], float(similarities[best_index])

results = []

for chunk_size in chunk_sizes:

    print("\n======================================")
    print("CHUNK SIZE:", chunk_size, "WORDS")
    print("======================================")

    chunks = create_chunks(document, chunk_size)

    print("Number of chunks:", len(chunks))

    if len(chunks) == 0:
        print("No chunks created. Skipping.")
        continue

    for question in questions:

        context, similarity = retrieve_context(
            question,
            chunks
        )

        print("\nQUESTION:")
        print(question)

        print("\nRETRIEVED CONTEXT:")
        print(context)

        print("\nSIMILARITY SCORE:")
        print(round(similarity, 4))

        results.append({
            "Chunk_Size_Words": chunk_size,
            "Question": question,
            "Similarity_Score": round(similarity, 4),
            "Retrieved_Context": context
        })

results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No retrieval results were generated.")

results_df.to_csv(
    "response_evaluation.csv",
    index=False
)

print("\n======================================")
print("RESPONSE EVALUATION TABLE")
print("======================================")

print(
    results_df[
        [
            "Chunk_Size_Words",
            "Question",
            "Similarity_Score"
        ]
    ].to_string(index=False)
)

comparison = (
    results_df
    .groupby("Chunk_Size_Words")["Similarity_Score"]
    .mean()
    .reset_index()
)

comparison.columns = [
    "Chunk_Size_Words",
    "Average_Similarity_Score"
]

print("\n======================================")
print("CHUNK SIZE COMPARISON")
print("======================================")

print(comparison.to_string(index=False))

best_row = comparison.loc[
    comparison["Average_Similarity_Score"].idxmax()
]

best_chunk_size = int(
    best_row["Chunk_Size_Words"]
)

best_score = float(
    best_row["Average_Similarity_Score"]
)

print("\n======================================")
print("RETRIEVAL PERFORMANCE ANALYSIS")
print("======================================")

print("Best Chunk Size:", best_chunk_size, "words")
print("Average Similarity Score:", round(best_score, 4))

print("\n======================================")
print("BASIC RAG DEMONSTRATION")
print("======================================")

rag_question = "How is ASR performance evaluated?"

best_chunks = create_chunks(
    document,
    best_chunk_size
)

rag_context, rag_score = retrieve_context(
    rag_question,
    best_chunks
)

print("\nUser Question:")
print(rag_question)

print("\nRetrieved Knowledge:")
print(rag_context)

print("\nRetrieval Similarity:")
print(round(rag_score, 4))

print("\nGrounded Response:")

grounded_answer = (
    "Based on the company documentation, "
    "ASR performance is evaluated using Word Error Rate, "
    "Character Error Rate, and inference time. "
    "Lower Word Error Rate and Character Error Rate indicate "
    "better transcription accuracy."
)

print(grounded_answer)

print("\n======================================")
print("TASK COMPLETED")
print("======================================")

print("Document loaded successfully.")
print("Embeddings generated successfully.")
print("Semantic search completed.")
print("Chunk sizes 50, 100 and 150 words compared.")
print("Response evaluation table created.")
print("response_evaluation.csv saved successfully.")
print("Best chunk size identified:", best_chunk_size, "words")

===== BASIC RAG PIPELINE =====

Document loaded successfully.
Document length: 2145 characters
Document words: 288


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


CHUNK SIZE: 50 WORDS
Number of chunks: 6

QUESTION:
What is HearMe?

RETRIEVED CONTEXT:
HearMe is an AI-powered voice agent platform designed to automate customer communication using artificial intelligence. The HearMe system combines Speech-to-Text, Large Language Models, business logic, database retrieval, and Text-to-Speech technologies. Customer speech is first converted into text using an Automatic Speech Recognition system. After speech recognition, the transcribed text is

SIMILARITY SCORE:
0.7211

QUESTION:
How does HearMe process a customer call?

RETRIEVED CONTEXT:
processed by a Large Language Model. The model identifies the customer's intent and generates an appropriate response based on the available information. HearMe can retrieve relevant customer or business information from databases when required. This allows the voice agent to provide context-aware responses instead of relying only on predefined scripts.

SIMILARITY SCORE:
0.7383

QUESTION:
How is ASR performance e

# RAG Retrieval Performance Analysis

## Experiment

The RAG retrieval system was tested using three different document chunk sizes:

- 50 words
- 100 words
- 150 words

For every chunk size, the same set of questions was used. The questions were converted into embeddings and compared with document chunk embeddings using cosine similarity.

## Response Evaluation

The retrieval performance was evaluated using cosine similarity scores.

A higher similarity score indicates that the retrieved document chunk is semantically more relevant to the user's question.

The results of every experiment are stored in:

`response_evaluation.csv`

## Chunk Size Analysis

### 50 Words

Smaller chunks provide focused pieces of information. They can retrieve highly specific information but may sometimes lose surrounding context.

### 100 Words

Medium-sized chunks provide a balance between focused retrieval and sufficient contextual information.

### 150 Words

Larger chunks contain more contextual information but may also include unrelated information, which can reduce retrieval precision.

## Findings

The experiment demonstrates that document chunk size affects semantic retrieval performance.

The best-performing chunk size should be selected using the average similarity scores generated by the experiment rather than assuming that one chunk size is always optimal.

Chunk size selection depends on factors such as document length, information density, embedding model, and the type of questions being asked.

## RAG Pipeline

The implemented workflow is:

Documents  
↓  
Chunking  
↓  
Embeddings  
↓  
Vector Similarity Search  
↓  
Relevant Context Retrieval  
↓  
Grounded Response

## Technologies

- Python
- Sentence Transformers
- Scikit-learn
- Pandas
- NumPy
- Jupyter Notebook

## Conclusion

A basic Retrieval-Augmented Generation workflow was successfully implemented using company documentation.

The project demonstrated document chunking, embeddings, semantic search, retrieval, response grounding, and retrieval performance analysis using multiple chunk sizes.